# TD4 — Feature engineering EEG pour l'estimation de la charge cognitive

## Contexte

Ce TD s'inscrit dans le projet fil rouge basé sur le papier :

**Multimodal Brain-Computer Interface for In-Vehicle Driver Cognitive Load Measurement: Dataset and Baselines**.

Le papier introduit le dataset **CL-Drive**, dans lequel des signaux EEG, ECG, EDA et Gaze sont enregistrés pendant une tâche de conduite simulée. Les scores de charge cognitive sont collectés toutes les **10 secondes**. Après prétraitement, les signaux sont segmentés en fenêtres de 10 s, puis transformés en caractéristiques numériques pour entraîner des modèles de classification.

Dans ce TD, on se concentre uniquement sur les **features EEG**.

---

## Objectifs pédagogiques

À la fin du TD, vous devez être capables de :

1. expliquer pourquoi on transforme un signal EEG en vecteur de caractéristiques ;
2. distinguer les features temporelles, fréquentielles et non linéaires ;
3. expliquer le principe de la densité spectrale de puissance, ou PSD ;
4. calculer des puissances par bande EEG ;
5. interpréter l'entropie spectrale ;
6. expliquer les paramètres de Hjorth ;
7. comprendre le principe de la complexité de Lempel-Ziv ;
8. comprendre l'idée de la dimension fractale de Higuchi ;
9. structurer une fonction complète d'extraction de features EEG ;
10. préparer une matrice de features pour la classification.

---

## Features EEG ciblées

Le papier regroupe les features EEG suivantes :

| Famille | Features |
|---|---|
| PSD | puissance absolue, moyenne, maximale, minimale et médiane |
| Entropie spectrale | entropie calculée à partir de la PSD normalisée |
| Hjorth | mobility et complexity |
| Lempel-Ziv | complexité d'une séquence binarisée |
| Higuchi | dimension fractale |
| Statistiques temporelles | moyenne, minimum, maximum, médiane, variance, écart-type |

Dans ce TD, on adopte une version complète :

- PSD calculée dans les cinq bandes EEG : delta, theta, alpha, beta, gamma ;
- entropie spectrale calculée dans les cinq bandes ;
- features non linéaires calculées sur le segment temporel ;
- statistiques temporelles calculées sur le segment.

On obtient donc :

$$
5 \text{ bandes} \times 5 \text{ descripteurs PSD} = 25
$$

$$
5 \text{ entropies spectrales} = 5
$$

$$
2 \text{ Hjorth} + 1 \text{ Lempel-Ziv} + 1 \text{ Higuchi} + 6 \text{ statistiques} = 10
$$

Soit au total :

$$
25 + 5 + 10 = 40 \text{ features par canal EEG}
$$

## Questions de compréhension

### Question 1

Pourquoi ne donne-t-on pas directement le signal EEG brut à un classifieur classique comme LDA, SVM ou Random Forest ?

### Réponse 

Dans le papier, il est indiqué que les chercheurs ont "entraîné les modèles à la fois sur des caractéristiques définies manuellement et sur des données brutes". Cependant, il existe des inconvénients à donner un signal EEG brut directemement à un classifieur classique :
- Trop de donneés d'entrée : une fenêtre de 10 secondes à 256 Hz contient 2560 échantillons par canal, soit 10 240 échantillons pour les 4 canaux. Les classifieurs classiques ne sont pas pensés pour traiter un aussi grand nombre de données. Par exemple, un article de [ScienceDirect](https://www.sciencedirect.com/science/article/pii/S0925231207002962) indique : "normal SVM is not suitable for classification of large data sets, because the training complexity of SVM is highly dependent on the size of data set".
- Trop de bruit et d'artefacts : le papier indique clairement ce problème, notamment avec la phrase "To remove noise and artifacts from EEG, we used a Butterworth 2nd order bandpass filter with a passband frequency of 0.4 to 75 Hz".

De manière générale, un classifieur classique attend des données de taille fixe pour simplifier le traitement. Le fait de donner le signal EEG brut reviendrait à surcharger le modèle d'informations redondantes et bruitées.

### Question 2

Pourquoi les features fréquentielles sont-elles particulièrement importantes en EEG ?

### Réponse 

Les features fréquentielles sont particulièrement importantes en EEG car, selon le papier, elles ont un lien direct avec l'activité cérébrale. En effet, "This property of EEG allows it to capture changes in brain activity while experiencing variations in cognitive load". La PSD permet de mesurer ces variations en mesurant la puissance du signal dans chaque bande fréquentielle (delta, theta, alpha, beta, gamma). De son côté, l'entropie spectrale complète cette analyse en mesurant la complexité de leur répartition. Les features fréquentielles sont donc utiles pour détecter les changements de charge cognitive du conducteur.

### Question 3

Pourquoi faut-il calculer les features séparément sur chaque canal EEG ?

### Réponse 

Il faut calculer les features séparément sur chaque canal EEG car chaque canal est positionné sur une région différente du cerveau. Les électrodes frontales et temporales ne capturent donc pas la même activité cérébrale. Il faut calculer les features séparément pour conserver l'information spatiale. Sans cela, il serait plus difficile d'estimer la charge cognitive.

## 1. Bandes fréquentielles EEG

Les signaux EEG sont souvent analysés par bandes de fréquence.

| Bande | Intervalle utilisé dans ce TD | Interprétation générale |
|---|---:|---|
| Delta | 0.5–4 Hz | activité lente |
| Theta | 4–8 Hz | attention, mémoire de travail, somnolence selon contexte |
| Alpha | 8–12 Hz | relaxation, inhibition, yeux fermés |
| Beta | 12–30 Hz | activité mentale, attention, activité motrice |
| Gamma | 30–75 Hz | activité rapide, intégration, mais sensible aux artefacts musculaires |

Dans le papier, la bande gamma va jusqu'à 75 Hz. 

### Question 4

Pourquoi peut-on limiter la bande gamma à 45 Hz dans certaines implémentations ?

### Réponse 

On peut limiter la bande gamma à 45 Hz dans certaines implémentations car cela réduit l'influence des artéfacts EMG (électromyogramme). Ce n'est pas indiqué explicitement dans le papier, mais selon l'article [High-frequency brain activity and muscle artifacts in MEG/EEG: a review and recommendations](https://pmc.ncbi.nlm.nih.gov/articles/PMC3625857/), "la contamination du spectre de fréquences commence autour de 20 Hz, de sorte qu'à 40 Hz, la puissance était environ 5 fois plus élevée dans l'état non paralysé, tandis qu'à 80 Hz, elle était environ dix fois plus élevée". La limitation de la bande gamma à 45 Hz permet donc de réduire l'influence des artéfacts musculaires car ils sont particulièrement présents au-delà de 40 Hz.

## 2. Méthodologie d'implémentation 

Dans ce TD, l'objectif est de comprendre puis implémenter les fonctions essentielles.

Pour chaque fonction, vous aurez :

- une explication théorique ;
- l'algorithme ;
- les fonctions Python recommandées ;
- les paramètres importants ;
- des questions de vérification.

Les bibliothèques utiles sont :

| Objectif | Bibliothèque | Fonctions utiles |
|---|---|---|
| Tableaux numériques | NumPy | `np.array`, `np.mean`, `np.var`, `np.diff`, `np.median` |
| Données tabulaires | pandas | `pd.DataFrame`, `pd.read_csv`, `to_csv` |
| PSD | scipy.signal | `welch` |
| Entropie | scipy.stats | `entropy` |
| Visualisation | matplotlib | `plt.plot`, `plt.semilogy`, `plt.bar` |

### Travail à réaliser

Vous devez construire progressivement les fonctions suivantes :

1. `compute_psd_band_features(signal, fs)` ;
2. `compute_spectral_entropy_bands(signal, fs)` ;
3. `compute_hjorth(signal)` ;
4. `lempel_ziv_complexity(signal)` ;
5. `higuchi_fd(signal, kmax=10)` ;
6. `compute_raw_features(signal)` ;
7. `extract_eeg_features(signal, fs)` ;
8. une boucle permettant d'extraire les features sur des fenêtres de 10 secondes.

In [9]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.stats import entropy

FS = 256
WINDOW_SEC = 10
WINDOW_SAMPLES = FS * WINDOW_SEC

EEG_BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 31),
    "gamma": (31, 75),
}

print("Nombre d'échantillons par fenêtre :", WINDOW_SAMPLES)
print("Bandes EEG :", EEG_BANDS)

Nombre d'échantillons par fenêtre : 2560
Bandes EEG : {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 12), 'beta': (12, 31), 'gamma': (31, 75)}


## 3. Exemples pédagogiques sur signaux connus

Avant d'appliquer les features à l'EEG réel, on commence par des signaux connus.

On va comparer :

1. une sinusoïde à 2 Hz, principalement dans la bande delta ;
2. une sinusoïde à 10 Hz, principalement dans la bande alpha ;
3. une sinusoïde à 20 Hz, principalement dans la bande beta ;
4. un bruit blanc, dont l'énergie est plus répartie ;
5. un mélange de plusieurs composantes.

### Objectif pédagogique

Vérifier que les features donnent des résultats cohérents :

- une sinusoïde pure doit avoir une PSD concentrée ;
- le bruit doit avoir une entropie spectrale plus élevée ;
- une sinusoïde alpha doit avoir une puissance alpha dominante.

In [2]:
# Cellule d'aide : génération de signaux synthétiques pour les tests pédagogiques.
# Vous pouvez l'utiliser pour tester vos futures fonctions de features.

def generate_synthetic_signals(fs=FS, duration=10, random_state=0):
    rng = np.random.default_rng(random_state)
    t = np.arange(0, duration, 1/fs)
    signals = {
        "delta_2Hz": np.sin(2*np.pi*2*t),
        "alpha_10Hz": np.sin(2*np.pi*10*t),
        "beta_20Hz": np.sin(2*np.pi*20*t),
        "white_noise": rng.normal(0, 1, size=len(t)),
        "mixed": 0.8*np.sin(2*np.pi*6*t) + 0.5*np.sin(2*np.pi*10*t) + 0.2*rng.normal(0, 1, size=len(t)),
    }
    return t, signals

t, synthetic_signals = generate_synthetic_signals()

## 4. Feature PSD : densité spectrale de puissance

### Principe théorique

La **PSD**, ou densité spectrale de puissance, indique comment la puissance du signal est répartie selon la fréquence.

Pour un segment EEG $x[n]$, on estime la PSD avec la méthode de Welch. L'idée est de :

1. découper le signal en sous-fenêtres ;
2. calculer un spectre sur chaque sous-fenêtre ;
3. moyenner les spectres pour obtenir une estimation plus stable.

En Python, on utilise généralement :

```python
freqs, psd = scipy.signal.welch(signal, fs=fs, nperseg=...)
```

### Paramètres recommandés

- `fs=256` pour CL-Drive ;
- `nperseg=fs*2` pour une fenêtre Welch de 2 secondes ;
- si le segment est court, prendre `nperseg=min(len(signal), fs*2)` ;
- sélectionner ensuite les fréquences appartenant à une bande donnée à l’aide d’un masque booléen (utiliser le vecteur `freqs` retourné `scipy.signal.welch`).

### Features PSD pour chaque bande

Pour chaque bande EEG, calculer :

1. puissance absolue : somme de la PSD dans la bande ;
2. puissance moyenne ;
3. puissance maximale ;
4. puissance minimale ;
5. puissance médiane.

### Algorithme

1. calculer `freqs, psd` avec Welch ;

Pour chaque bande $[f_{min}, f_{max}]$ :

   
2. sélectionner les indices tels que $f_{min} \le f < f_{max}$ ;
3. extraire `band_psd` ;
4. calculer `sum`, `mean`, `max`, `min`, `median` ;
5. stocker les résultats dans un dictionnaire.

### Questions

1. Quelle bande doit dominer pour un signal sinusoïdal à 10 Hz ?
2. Pourquoi la PSD est-elle plus stable avec Welch qu'avec un simple spectre FFT ?
3. Que se passe-t-il si la bande sélectionnée ne contient aucune fréquence ?

### Réponses 

1. La bande qui doit dominer pour un signal sinusoïdal à 10 Hz est la bande alpha. En effet, cette dernière couvre de 8 Hz à 13 Hz.
2. Le papier indique "To calculate this feature we use the Welch's method from 0.5Hz to 75 Hz frequency". Or, ce n'est pas expliqué pourquoi Welch est choisi. La raison est que la méthode de Welch découpe le signal en plusieurs sous-fenêtres, calcule un spectre sur chacune, puis fait la moyenne des résultats. À l'inverse, la FFT serait très sensible aux variations locales.
3. Si la bande sélectionnée ne contient aucune fréquence, alors aucun des points du spectre calculé par Welch ne tombera dans l'intervalle de la bande. On se retrouvera alors avec un tableau vide et des calculs de somme, moyenne, etc. qui retourneront 0 ou une valeur nulle.

In [3]:
# À faire dans votre notebook de travail : implémenter compute_welch_psd() et compute_psd_band_features().
# Indication : utiliser scipy.signal.welch, puis sélectionner chaque bande avec un masque fréquentiel.
def compute_welch_psd(signal, fs=FS):
  nperseg = min(len(signal), fs * 2)
  freqs, psd = welch(signal, fs=fs, nperseg=nperseg)
  return freqs, psd

def compute_psd_band_features(signal, fs=FS):
  freqs, psd = compute_welch_psd(signal, fs)
  features = {}

  for band_name, (fmin, fmax) in EEG_BANDS.items():
    idx = (freqs >= fmin) & (freqs <= fmax)
    band_psd = psd[idx]

    if len(band_psd) == 0:
      features[f"{band_name}_abs"] = 0.0
      features[f"{band_name}_mean"] = 0.0
      features[f"{band_name}_max"] = 0.0
      features[f"{band_name}_min"] = 0.0
      features[f"{band_name}_median"] = 0.0
    else:
      features[f"{band_name}_abs"] = np.sum(band_psd)
      features[f"{band_name}_mean"] = np.mean(band_psd)
      features[f"{band_name}_max"] = np.max(band_psd)
      features[f"{band_name}_min"] = np.min(band_psd)
      features[f"{band_name}_median"] = np.median(band_psd)

  return features

## 5. Entropie spectrale

### Principe théorique

L'entropie spectrale mesure la dispersion de l'énergie dans le domaine fréquentiel.

On calcule d'abord la PSD dans une bande, puis on la normalise pour obtenir une distribution :

$$
p_i = \frac{PSD_i}{\sum_j PSD_j}
$$

L'entropie de Shannon est ensuite :

$$
H = - \sum_i p_i \log(p_i)
$$


### Fonction Python utile

```python
from scipy.stats import entropy
```

`entropy(p)` calcule directement l'entropie de Shannon si `p` est une distribution normalisée.

### Questions

1. Pourquoi faut-il normaliser la PSD avant de calculer l'entropie ?
2. Quel signal devrait avoir l'entropie spectrale la plus élevée : une sinusoïde pure ou un bruit blanc ?
3. Pourquoi l'entropie spectrale peut-elle être utile pour caractériser la complexité d'un EEG ?

### Réponses 

1. Il faut normaliser la PSd avant de calculer l'entropie car l'entropie de Shannon a besoin de proportions en entrée, pas de valeurs brutes. On divise donc chaque valeur par la somme totale pour savoir non plus "combien de puissance dans cette bande", mais "quelle part de la puissance totale est dans cette bande".
2. Entre une sinusoïde pure et un bruit blanc, le signal qui devrait avoir l'entropie spectrale la plus élevée est le bruit blanc. Une sinusoïde pure met toute son énergie sur une seule fréquence, donc sa PSD normalisée ressemble à un pic à 1 et des 0 partout ailleurs, ce qui donne une entropie très faible. Le bruit blanc étale son énergie partout de façon uniforme, donc toutes les proportions sont à peu près égales, et l'entropie est maximale.
3. L'entropie spectrale peut être utile pour caractériser la complexité d'un EEG car elle mesure à quel point l'énergie du signal est répartie sur les différentes fréquences. Un EEG simple et régulier concentre son énergie sur peu de fréquences, donc l'entropie est faible. Un EEG complexe, comme en situation de forte charge cognitive, active plusieurs bandes en même temps, donc l'entropie est élevée.

In [4]:
# À vous d'implémenter compute_spectral_entropy_bands().
def compute_spectral_entropy_bands(signal, fs=FS):
    freqs, psd = compute_welch_psd(signal, fs)
    features = {}

    for band_name, (fmin, fmax) in EEG_BANDS.items():
        idx = (freqs >= fmin) & (freqs < fmax)
        band_psd = psd[idx]

        if len(band_psd) == 0 or np.sum(band_psd) == 0:
            features[f"{band_name}_entropy"] = 0.0
        else:
            band_psd_norm = band_psd / np.sum(band_psd)
            features[f"{band_name}_entropy"] = entropy(band_psd_norm)

    return features

## 6. Paramètres de Hjorth : mobility et complexity

Les paramètres de Hjorth sont des descripteurs temporels utilisés pour caractériser la dynamique d'un signal.

### Mobility

La mobility mesure approximativement la fréquence moyenne du signal :

$$
Mobility(x) = \sqrt{\frac{Var(\Delta x)}{Var(x)}}
$$

où $\Delta x$ est la dérivée discrète du signal, généralement calculée avec `np.diff(x)`.

### Complexity

La complexity mesure la variation de la mobility entre le signal et sa dérivée :

$$
Complexity(x) = \frac{Mobility(\Delta x)}{Mobility(x)}
$$


### Questions

1. Que vaut approximativement la variance d'un signal constant ?
2. Pourquoi faut-il gérer le cas où `Var(x)=0` ?
3. Entre une sinusoïde lisse et un bruit blanc, lequel devrait avoir une complexity plus élevée ?

### Réponses 

1. La variance d'un signal constant vaut approximativement 0. La variance est l'écart des valeurs par rapport à leur moyenne, donc si toutes les valeurs sont identiques, il n'y a aucun écart et est nulle.
2. Il faut gérer le cas où Var(x)=0 car cette expression est utilisée dans le calcul de la mobility, au niveau du dénominateur. Si Var(x)=0, alors il y a une division par 0, ce qui est impossible. Cela pourrait poser des problèmes dans l'algorithme Python.
3. Entre une sinusoïde lisse et un bruit blanc, c'est le bruit blanc qui devrait avoir une complexity plus élevée. La complexity mesure à quel point la forme du signal change entre le signal et sa dérivée. Une sinusoïde est très régulière : sa dérivée est aussi une sinusoïde, donc la mobility ne change pas beaucoup entre les deux. Le bruit blanc est irrégulier et change brusquement à chaque échantillon, donc sa dérivée est encore plus irrégulière, ce qui donne une complexity plus élevée.

In [5]:
# À vous d'implémenter compute_hjorth().
def compute_hjorth(signal):
    signal = np.array(signal, dtype=float)
    
    var_x = np.var(signal)
    if var_x == 0:
        return {"hjorth_mobility": 0.0, "hjorth_complexity": 0.0}

    dx = np.diff(signal)
    var_dx = np.var(dx)
    mobility = np.sqrt(var_dx / var_x)

    var_ddx = np.var(np.diff(dx))
    if var_dx == 0:
        return {"hjorth_mobility": mobility, "hjorth_complexity": 0.0}

    mobility_dx = np.sqrt(var_ddx / var_dx)
    complexity = mobility_dx / mobility

    return {"hjorth_mobility": mobility, "hjorth_complexity": complexity}

## 7. Complexité de Lempel-Ziv

### Principe théorique

La complexité de Lempel-Ziv mesure le nombre de motifs nouveaux rencontrés dans une séquence.

Comme l'algorithme s'applique à une séquence symbolique, un signal EEG réel doit d'abord être transformé en séquence binaire.

Une méthode simple consiste à binariser le signal par rapport à sa médiane :

$$
b[n] =
\begin{cases}
1, & x[n] > median(x) \\
0, & x[n] \le median(x)
\end{cases}
$$


### Paramètres importants

- seuil de binarisation : médiane ou moyenne ;
- normalisation de la complexité pour comparer des segments de même ou de différente longueur ;
- gestion des segments constants.

### Questions

1. Pourquoi faut-il binariser le signal avant de calculer Lempel-Ziv ?
2. Pourquoi la médiane est-elle un seuil intéressant ?
3. Quel signal devrait avoir une complexité plus élevée : une sinusoïde pure ou un bruit blanc ?
4. Pourquoi normaliser la complexité par la longueur de la séquence ?

### Réponses 

1. Il faut binariser le signal avant de calculer Lempel-Ziv car l'algorithme travaille sur des séquences de symboles discrets, pas sur des valeurs continues. Comme un signal EEG est une suite de valeurs réelles, on ne peut pas l'utiliser ainsi. On le transforme donc en une suite de 0 et de 1, ce qui permet à l'algorithme de repérer et compter les nouveaux motifs qui apparaissent.
2. La médiane est un seuil intéressant car elle coupe le signal en deux : la moitié des valeurs seront au-dessus, l'autre moitié en dessous. On obtient donc toujours autant de 0 que de 1 dans la séquence binarisée. La moyenne pourrait aussi fonctionner, mais si le signal est asymétrique, elle peut être tirée vers le haut ou vers le bas par quelques valeurs extrêmes, ce qui déséquilibrerait la séquence binarisée.
3. Le bruit blanc devrait avoir une complexité de Lempel-Ziv plus élevée. Une sinusoïde binarisée produit une séquence très régulière qui alterne entre 0 et 1 de façon périodique, donc peu de nouveaux motifs apparaissent. Le bruit blanc binarisé produit une séquence imprévisible où de nouveaux motifs apparaissent en permanence, ce qui donne une complexité plus élevée.
4. Il faut normaliser la complexité par la longueur de la séquence car plus un signal est long, plus il contient de motifs, donc plus sa complexité sera élevée. Sans normalisation, on ne pourrait pas comparer deux segments de durées différentes. En divisant par la longueur, on ramène tout à la même échelle.

In [6]:
# À vous d'implémenter lempel_ziv_complexity().
def lempel_ziv_complexity(signal):
    signal = np.array(signal, dtype=float)
    
    # Binarisation par la médiane
    median = np.median(signal)
    binary = (signal > median).astype(int)
    
    # Algorithme de Lempel-Ziv : compter les nouveaux motifs
    n = len(binary)
    complexity = 1
    i, k, l = 0, 1, 1

    while k + l <= n:
        if binary[i + l - 1] == binary[k + l - 1]:
            l += 1
        else:
            i = max(i + 1, k - complexity + 1)  # reculer dans la séquence
            if i < k:
                l = 1
            else:
                complexity += 1
                k = k + l
                i = 0
                l = 1

    # Normalisation par la longueur
    if n == 0 or np.log2(n) == 0:
        return {"lempel_ziv": 0.0}
    
    lzc = complexity / (n / np.log2(n))
    return {"lempel_ziv": lzc}  # 1 feature

## 8. Dimension fractale de Higuchi

### Principe théorique

La dimension fractale de Higuchi cherche à mesurer la complexité géométrique d'un signal temporel.

Un signal très lisse ressemble davantage à une courbe régulière. Un signal très irrégulier ou bruité présente une trajectoire plus complexe.

L'algorithme de Higuchi construit plusieurs sous-séquences avec différents pas $k$, mesure leur longueur moyenne $L(k)$, puis estime une pente dans un espace logarithmique.

### Paramètre important

- `kmax` : pas maximal testé.

Pour un segment de 10 secondes à 256 Hz, une valeur pédagogique simple est :

```python
kmax = 10
```

Une valeur trop faible peut donner une estimation instable ; une valeur trop élevée augmente le coût de calcul.

### Questions

1. Que cherche à mesurer la dimension fractale de Higuchi ?
2. Pourquoi un signal bruité peut-il avoir une dimension fractale plus élevée qu'une sinusoïde ?
3. Quel est le rôle du paramètre `kmax` ?
4. Pourquoi faut-il éviter de calculer un logarithme de zéro ?

### Réponses 

1. La dimension fractale de Higuchi quantifie à quel point un signal est "irrégulier" ou "rugueux". Pour un signal EEG, cela permet de capturer des changements dans l'activité cérébrale qui ne seraient pas visibles avec des méthodes classiques dans le domaine fréquentiel.

2. Plus un signal est complexe, plus sa dimension fractale est élevée. Ainsi, une sinusoïde est un signal régulier et prévisible donc sa dimension fractale est proche de 1 alors Un signal bruité lui est beaucoup plus irrégulier donc sa dimension fractale se rapproche de 2.

3. kmax est le nombre maximum de pas k qu'on va tester pour construire les sous-séquences. Si kmax est trop petit, on analyse le signal sur trop peu d'échelles et la pente estimée dans l'espace logarithmique sera peu fiable. Si kmax est trop grand, les sous-séquences deviennent trop courtes et contiennent trop peu de points pour être représentatives.

4. Dans le calcul de Higuchi on fait une régression linéaire sur des valeurs logarithmiques. Si une des longueurs calculées vaut zéro alors log(0) est indéfini et le calcul se termine en erreur. C'est pour ça qu'il faut vérifier que le signal n'a pas de segments plats ou de valeurs manquantes avant de lancer le calcul.

In [17]:
# À vous d'implémenter higuchi_fd().

import numpy as np

def higuchi_fd(x, kmax):
    N = len(x)
    L = []

    for k in range(1, kmax + 1):
        Lk = []
        for m in range(1, k + 1):
            # Construction de la sous-séquence
            Lmk = 0
            n_max = int(np.floor((N - m) / k))
            for i in range(1, n_max):
                Lmk += abs(x[m + i*k - 1] - x[m + (i-1)*k - 1])
            Lmk = Lmk * (N - 1) / (k * n_max)
            Lk.append(Lmk)
        L.append(np.mean(Lk))

    # Régression linéaire dans l'espace logarithmique
    log_k = np.log(range(1, kmax + 1))
    log_L = np.log(L)
    hfd = np.polyfit(log_k, log_L, 1)[0]

    return hfd

## 9. Statistiques temporelles du signal brut

Les statistiques temporelles simples fournissent des informations directes sur l'amplitude et la variabilité du signal.

Features demandées :

1. moyenne ;
2. minimum ;
3. maximum ;
4. médiane ;
5. variance ;
6. écart-type.

### Fonctions Python utiles

| Feature | Fonction NumPy |
|---|---|
| moyenne | `np.mean` |
| minimum | `np.min` |
| maximum | `np.max` |
| médiane | `np.median` |
| variance | `np.var` |
| écart-type | `np.std` |


In [ ]:
# À vous d'implémenter compute_raw_features().

def statistical_features(x):
    features = {
        'mean': np.mean(x),
        'min': np.min(x),
        'max': np.max(x),
        'median': np.median(x),
        'var': np.var(x),
        'std': np.std(x)
    }
    return features

def compute_raw_features(x, kmax=8):
    features = []

    # Features statistiques
    stats = statistical_features(x)
    features.extend(stats.values())

    # Dimension fractale de Higuchi
    hfd = higuchi_fd(x, kmax)
    features.append(hfd)

    return np.array(features)

## 10. Fonction complète d'extraction des 40 features EEG

À ce stade, on peut regrouper toutes les familles de features dans une seule fonction.

### Entrée

Un segment EEG 1D correspondant à :

- un canal ;
- une fenêtre de 10 secondes ;
- 2560 échantillons si `fs=256 Hz`.

### Sortie

Un dictionnaire de **40 features**.

### Organisation recommandée

1. convertir le signal en tableau NumPy ;
2. remplacer les valeurs manquantes par 0 ou par une stratégie décidée en amont ;
3. calculer les features PSD par bande ;
4. calculer les entropies spectrales ;
5. calculer Hjorth ;
6. calculer Lempel-Ziv ;
7. calculer Higuchi ;
8. calculer les statistiques temporelles ;
9. fusionner les dictionnaires.

### Questions

1. Pourquoi la fonction doit-elle retourner un dictionnaire plutôt qu'une simple liste ?
2. Pourquoi est-il important de conserver des noms de colonnes explicites ?
3. Combien de features doit retourner la fonction pour un canal ?
4. Si on a 4 canaux et qu'on concatène toutes les features, combien de features obtient-on par segment ?

### Réponses 

1. Un dictionnaire associe chaque valeur à un nom de feature. Avec une liste, on ne sait plus ce que représente la valeur à l'indice 12 par exemple. Un dictionnaire permet de convertir facilement le résultat en DataFrame avec des noms de colonnes lisibles.
2. Sans nom explicite, il est impossible de savoir quelle colonne correspond à quelle feature. Si on veut analyser l'importance d'une feature, on a besoin de savoir si on regarde la puissance alpha ou l'écart-type du signal. Des noms comme alpha_mean ou hjorth_mobility sont bien plus explicites que feature_12.
3. La fonction doit retourner 40 features par canal : 25 features PSD (5 bandes x 5 descripteurs), 5 entropies spectrales, 2 paramètres de Hjorth, 1 Lempel-Ziv, 1 Higuchi, et 6 statistiques temporelles.
4. On obtient 160 features par segment : 40 features par canal x 4 canaux.

In [14]:
# À vous d'implémenter extract_eeg_features() et de vérifier qu'elle retourne 40 features.

def extract_eeg_features(x, fs=256, kmax=8):
    x = np.array(x, dtype=float)

    # Remplacement des valeurs manquantes
    x = np.nan_to_num(x, nan=0.0)

    features = {}

    # 1. PSD par bande
    bands = {
        'delta': (0.5, 4),
        'theta': (4, 8),
        'alpha': (8, 12),
        'beta':  (12, 31),
        'gamma': (31, 75)
    }

    freqs, psd = welch(x, fs=fs, nperseg=fs*2)

    for band_name, (fmin, fmax) in bands.items():
        idx = np.where((freqs >= fmin) & (freqs < fmax))
        band_psd = psd[idx]
        features[f'{band_name}_abs']    = np.sum(band_psd)
        features[f'{band_name}_mean']   = np.mean(band_psd)
        features[f'{band_name}_max']    = np.max(band_psd)
        features[f'{band_name}_min']    = np.min(band_psd)
        features[f'{band_name}_median'] = np.median(band_psd)

    # 2. Entropie spectrale (une par bande)
    psd_norm = psd / np.sum(psd)
    for band_name, (fmin, fmax) in bands.items():
        idx = np.where((freqs >= fmin) & (freqs < fmax))
        p = psd_norm[idx]
        p = p[p > 0]
        features[f'{band_name}_entropy'] = -np.sum(p * np.log(p))

    # 3. Hjorth
    features['hjorth_mobility'], features['hjorth_complexity'] = compute_hjorth(x)

    # 4. Lempel-Ziv
    features['lempel_ziv'] = lempel_ziv_complexity(x)

    # 5. Higuchi
    features['higuchi'] = higuchi_fd(x, kmax)

    # 6. Statistiques temporelles
    features['mean']   = np.mean(x)
    features['min']    = np.min(x)
    features['max']    = np.max(x)
    features['median'] = np.median(x)
    features['var']    = np.var(x)
    features['std']    = np.std(x)

    # Vérification
    assert len(features) == 40, f"Nombre de features incorrect : {len(features)}"

    return features

## 11. Comparaison des features sur signaux connus

On applique maintenant la fonction complète aux signaux synthétiques.

### Objectif

Vérifier que :

- `alpha_10Hz` a une puissance alpha élevée ;
- `beta_20Hz` a une puissance beta élevée ;
- `white_noise` a une entropie spectrale élevée ;
- les features non linéaires augmentent généralement avec l'irrégularité.

In [18]:
# À vous d'appliquer vos fonctions aux signaux synthétiques et d'interpréter les résultats.
# Application de extract_eeg_features() à chaque signal synthétique
results = {}
for name, signal in synthetic_signals.items():
    results[name] = extract_eeg_features(signal, fs=FS)

# Affichage sous forme de DataFrame pour comparer
df_results = pd.DataFrame(results).T

# On sélectionne les features les plus parlantes pour la vérification
features_to_check = [
    'alpha_abs', 'alpha_mean',       # doit être élevé pour alpha_10Hz
    'beta_abs', 'beta_mean',         # doit être élevé pour beta_20Hz
    'delta_abs', 'delta_mean',       # doit être élevé pour delta_2Hz
    'delta_entropy', 'alpha_entropy',
    'beta_entropy', 'gamma_entropy', # doit être élevé pour white_noise
    'higuchi',                       # doit être élevé pour white_noise
    'lempel_ziv',                    # doit être élevé pour white_noise
]

print(df_results[features_to_check].round(4))

            alpha_abs alpha_mean  beta_abs beta_mean delta_abs delta_mean  \
delta_2Hz         0.0        0.0       0.0       0.0       1.0   0.142857   
alpha_10Hz        1.0      0.125       0.0       0.0       0.0        0.0   
beta_20Hz         0.0        0.0       1.0  0.026316       0.0        0.0   
white_noise  0.060019   0.007502  0.294935  0.007761  0.048637   0.006948   
mixed        0.252976   0.031622  0.012259  0.000323  0.002227   0.000318   

            delta_entropy alpha_entropy beta_entropy gamma_entropy   higuchi  \
delta_2Hz        0.867563           0.0          0.0           0.0   -0.0055   
alpha_10Hz            0.0      0.867563          0.0           0.0 -0.073963   
beta_20Hz             0.0           0.0     0.867563           0.0 -0.324537   
white_noise      0.135994      0.165116     0.805343      1.912526 -1.002627   
mixed            0.018187      0.586211     0.100282      0.219207 -0.457211   

                                      lempel_ziv  
delta

## 12. Application aux signaux EEG du dataset CL-Drive

Après les tests pédagogiques, les mêmes fonctions doivent être appliquées aux signaux EEG prétraités.

### Hypothèse de structure des fichiers

On suppose que les fichiers EEG prétraités sont des fichiers CSV contenant :

- une colonne `Timestamp` ;
- une colonne par canal EEG, par exemple `AF7`, `AF8`, `TP9`, `TP10`.

Exemple de structure :

| Timestamp | AF7 | AF8 | TP9 | TP10 |
|---:|---:|---:|---:|---:|
| 0.000 | ... | ... | ... | ... |
| 0.004 | ... | ... | ... | ... |

### Algorithme d'extraction sur un fichier

1. lire le fichier CSV avec `pd.read_csv` ;
2. identifier les colonnes EEG ;
3. découper le signal en fenêtres de 10 secondes ;
4. pour chaque fenêtre :
   - extraire les 2560 échantillons ;
   - pour chaque canal, calculer les 40 features ;
   - stocker les métadonnées : sujet, fichier, fenêtre, temps début, temps fin, canal ;
5. construire un `DataFrame` ;
6. sauvegarder le résultat en CSV.

### Question

Pourquoi faut-il conserver les colonnes `Participant`, `File`, `Window`, `Channel`, `Start_Time` et `End_Time` avec les features ?

In [ ]:
# À vous d'écrire une fonction d'extraction des features depuis un DataFrame EEG.

## 13. Traitement par lot des fichiers EEG prétraités

Dans le projet, les fichiers prétraités peuvent être organisés par sujet.

Exemple :

```text
Data/
|----EEG/
    ├── Participant_ID1/
    │   ├── filtered_scenario_1.csv
    │   ├── filtered_scenario_2.csv
    │   └── ...
    ├── Participant_ID2/
    │   └── ...
|----EDA
|----ECG
|----Gaze
|----Labels
```

### Algorithme par lot

1. parcourir les dossiers sujets ;
2. sélectionner uniquement les fichiers `filtered_*.csv` ;
3. lire chaque fichier ;
4. extraire les features fenêtre par fenêtre ;
6. sauvegarder un fichier CSV de features par sujet.

### Remarque importante

Les labels PAAS ne se trouvent pas dans le même fichier que les signaux. Une étape d’association entre les features et les labels est donc nécessaire dans un second temps : il faut relier les scores de charge cognitive aux intervalles temporels correspondants.

In [ ]:
# À vous d'adapter le traitement par lot à l'organisation réelle de vos fichiers.
# Exemple d'utilisation à adapter :
# batch_extract_eeg_features(
#     base_path="Data/EEG",
#     output_path="Data/EEG_Features_10s",
#     fs=256,
#     window_sec=10,
# )

## 14. Vérifications qualité des features

Avant de passer à la classification, il faut vérifier la qualité de la matrice de features.

### Vérifications recommandées

1. nombre de lignes cohérent avec le nombre de fenêtres et de canaux ;
2. absence de valeurs manquantes ;
3. absence de valeurs infinies ;
4. ordre de grandeur plausible ;
5. nombre de features égal à 40 par canal ;
6. conservation des métadonnées utiles ;
7. possibilité d'associer ensuite chaque fenêtre à un label.

### Questions

1. Pourquoi des valeurs `NaN` peuvent-elles apparaître dans les features ?
2. Pourquoi des valeurs infinies peuvent-elles apparaître ?
3. Que doit-on faire si un segment contient trop de valeurs manquantes ?
4. Pourquoi faut-il éviter de normaliser les features avant la séparation train/test ?

In [ ]:
# À vous d'écrire une fonction de contrôle qualité des features.